# `Feature Selection Techniques in Machine Learning`

Welcome! Today we’ll learn about **`Feature Selection`** – one of the most important steps in building a machine learning model. I’ll act as your teacher, explaining **why it is needed**, **how many types of techniques exist**, and then we’ll write code together in a Jupyter Notebook style. Let’s dive in!

---

## 1. Why Do We Need Feature Selection?

Imagine you are trying to predict whether a patient has a disease. You might have hundreds of measurements: blood pressure, age, weight, genetic markers, etc. Not all of them are useful. Some are **`irrelevant`**, some are **`redundant`**, and some may even introduce **`noise`**.


**Feature selection** helps us:

- **`Reduce overfitting`**: Less noise means the model generalizes better.
- **`Improve accuracy`**: Removing misleading data can boost performance.
- **`Reduce training time`**: Fewer features → faster computations.
- **`Enhance interpretability`**: A model with 5 features is easier to explain than one with 500.
- **`Avoid the curse of dimensionality`**: In high dimensions, data becomes sparse and distances lose meaning.

- Note:- The primary objectives are to preserve **`high predictive value`**, **`decrease dimensionality`**, and **`improve interpretability`**.


## 2. How Many Types of Feature Selection Techniques Are There?

There is no fixed number, but we usually classify them into **three main families** (some authors add a fourth “hybrid” category):

#### $\textbf{Filter Methods}$   
   - Rank features using statistical scores, independent of any ML model.  
   - Examples:
   
   1. $\textbf{Variance Threshold}$
      $$\sigma^2 = \frac{1}{N} \sum_{i=1}^{N} (x_i - \mu)^2$$
      
   2. $\textbf{Correlation (Pearson)}$
      $$r = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum (x_i - \bar{x})^2 \sum (y_i - \bar{y})^2}}$$
      
   3. $\textbf{Chi-Square }(\chi^2)$
      $$\chi^2 = \sum \frac{(O_i - E_i)^2}{E_i}$$      
      
   5. $\textbf{ANOVA (F-Statistic)}$
      $$F = \frac{\text{Variance Between Groups}}{\text{Variance Within Groups}} = \frac{MS_{\text{between}}}{MS_{\text{within}}}$$
      
   6. $\textbf{Mutual Information (MI)}$
      $$I(X; Y) = \sum_{x \in X} \sum_{y \in Y} p(x, y) \log \left( \frac{p(x, y)}{p(x)p(y)} \right)$$


2. #### $\textbf{Wrapper Methods} $ 
   - Use a predictive model to evaluate subsets of features.  
   - Examples: `Recursive Feature Elimination (RFE)`, `Forward/Backward Selection`, `Exhaustive Search`.

3. #### $ \textbf{Embedded Methods} $  
   - Feature selection happens **during** model training.  
   - Examples: `Lasso (L1 regularization)`, `Decision Tree / Random Forest importance`, `Elastic Net`.

4. #### $ \textbf{Hybrid Methods} $ (optional)  
   - Combine filter and wrapper approaches to get the best of both worlds.



## 3. Let's Start

We’ll use a classic dataset: **Breast Cancer Dataset** from `sklearn`. It has 30 numeric features and a binary target.

In [30]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [32]:
# Load data
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

In [34]:
# Let Check the Rows and Columns
print("Dataset shape:", X.shape)
print("Target classes:", np.unique(y))

Dataset shape: (569, 30)
Target classes: [0 1]


In [36]:
# Split into train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize (important for many feature selection methods)
#scaler = StandardScaler()
#X_train_scaled = scaler.fit_transform(X_train)
#X_test_scaled = scaler.transform(X_test)

## 4. Filter Methods

Filter methods evaluate features **without involving any ML model**. They are fast and good for a first pass.

### 4.1 Variance Threshold

Removes features with low variance (i.e., almost constant values). Those features carry little information.

In [41]:
from sklearn.feature_selection import VarianceThreshold

# Apply variance threshold (remove features with variance below 0.01)
selector = VarianceThreshold(threshold=0.1)
selector.fit(X_train)

# Get boolean mask of selected features
selected_mask = selector.get_support()
selected_features = X.columns[selected_mask]

print("Number of selected features:", len(selected_features))
print("Selected features:", selected_features.tolist())

Number of selected features: 11
Selected features: ['mean radius', 'mean texture', 'mean perimeter', 'mean area', 'texture error', 'perimeter error', 'area error', 'worst radius', 'worst texture', 'worst perimeter', 'worst area']


### 4.2 Correlation Matrix

Highly correlated features are redundant. We can visualise and remove one of each pair.

In [ ]:
# Compute correlation matrix on training data
corr_matrix = pd.DataFrame(X_train_scaled, columns=X.columns).corr().abs()

# Plot heatmap (optional, but nice to see)
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, cmap='coolwarm', annot=False)
plt.title('Feature Correlation Matrix')
plt.show()

# Find pairs with correlation > 0.9
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr_pairs = [(col, idx) for col in upper.columns for idx in upper.index if upper.loc[idx, col] > 0.9]
print("Highly correlated pairs (corr > 0.9):", high_corr_pairs)

> **Note:** Since our data is standardized, variance is around 1 for all features, so this may not remove many. On raw data it can be more useful.

### 4.3 Univariate Selection (SelectKBest)

We rank features using statistical tests and keep the top `k`. For classification we can use `f_classif` (ANOVA) or `chi2`.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

# Select top 10 features based on ANOVA F-value
k = 10
selector = SelectKBest(score_func=f_classif, k=k)
selector.fit(X_train_scaled, y_train)

# Get selected feature names and scores
scores = selector.scores_
feature_scores = pd.DataFrame({'Feature': X.columns, 'Score': scores})
top_features = feature_scores.nlargest(k, 'Score')

print("Top 10 features by ANOVA F-value:")
print(top_features)

# Transform the data to keep only selected features
X_train_selected = selector.transform(X_train_scaled)
X_test_selected = selector.transform(X_test_scaled)
print("New training shape:", X_train_selected.shape)

## 5. Wrapper Methods

Wrapper methods use a model to evaluate feature subsets. They are more accurate but computationally expensive.

### 5.1 Recursive Feature Elimination (RFE)

RFE recursively removes the least important features according to a model’s coefficients or feature importances.

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

# Use logistic regression as the estimator
model = LogisticRegression(max_iter=1000, random_state=42)

# Select top 10 features using RFE
rfe = RFE(estimator=model, n_features_to_select=10)
rfe.fit(X_train_scaled, y_train)

# Get selected feature names
selected_mask = rfe.support_
selected_features = X.columns[selected_mask]
print("RFE selected features:", selected_features.tolist())

# Transform data
X_train_rfe = rfe.transform(X_train_scaled)
X_test_rfe = rfe.transform(X_test_scaled)

### 5.2 Sequential Feature Selection (Forward/Backward)

`SequentialFeatureSelector` adds (forward) or removes (backward) features one by one, evaluating model performance at each step.

In [ ]:
from sklearn.feature_selection import SequentialFeatureSelector

# Forward selection with logistic regression, using 5-fold cross-validation
sfs = SequentialFeatureSelector(
    estimator=LogisticRegression(max_iter=1000, random_state=42),
    n_features_to_select=10,
    direction='forward',   # 'backward' for backward elimination
    cv=5,
    scoring='accuracy'
)
sfs.fit(X_train_scaled, y_train)

selected_features = X.columns[sfs.get_support()]
print("Sequential Forward selected features:", selected_features.tolist())

X_train_sfs = sfs.transform(X_train_scaled)
X_test_sfs = sfs.transform(X_test_scaled)

## 6. Embedded Methods

Embedded methods perform feature selection **as part of the model training process**.

### 6.1 Lasso Regularization (L1)

Lasso adds a penalty that can shrink coefficients to exactly zero – those features are effectively removed.

In [ ]:
from sklearn.linear_model import LogisticRegression

# Logistic Regression with L1 penalty (embedded feature selection)
l1_model = LogisticRegression(penalty='l1', solver='liblinear', C=0.1, random_state=42)
l1_model.fit(X_train_scaled, y_train)

# Features with non-zero coefficients
coef = l1_model.coef_.ravel()
selected_mask = coef != 0
selected_features = X.columns[selected_mask]
print("L1 selected features:", selected_features.tolist())
print("Number of selected features:", len(selected_features))

### 6.2 Tree-Based Feature Importance

Random Forest and Gradient Boosting provide feature importance scores derived from how often a feature is used to split nodes.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Train a Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_scaled, y_train)

# Get feature importances
importances = rf.feature_importances_
feature_importances = pd.DataFrame({'Feature': X.columns, 'Importance': importances})
feature_importances = feature_importances.sort_values('Importance', ascending=False)

# Plot top 10
plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_importances.head(10))
plt.title('Top 10 Feature Importances (Random Forest)')
plt.tight_layout()
plt.show()

# Select top 10 features
top10_features = feature_importances.head(10)['Feature'].tolist()
print("Top 10 features by Random Forest importance:", top10_features)

### 6.3 SelectFromModel

`SelectFromModel` uses any model’s feature importance or coefficients to select features above a threshold.

In [ ]:
from sklearn.feature_selection import SelectFromModel

# Use Random Forest importance to select features with importance > mean importance
sfm = SelectFromModel(estimator=RandomForestClassifier(n_estimators=100, random_state=42), threshold='mean')
sfm.fit(X_train_scaled, y_train)

selected_features = X.columns[sfm.get_support()]
print("SelectFromModel selected features:", selected_features.tolist())

X_train_sfm = sfm.transform(X_train_scaled)
X_test_sfm = sfm.transform(X_test_scaled)

## 7. Comparison of Techniques

| Method Type | Pros | Cons |
|-------------|------|------|
| **Filter** | Fast, model-agnostic, good for high-dimensional data | Ignores feature interactions, may select redundant features |
| **Wrapper** | Considers feature interactions, often better performance | Computationally expensive, risk of overfitting to the model |
| **Embedded** | Integrated with training, good trade-off | Specific to the model used, less flexible |

**When to use what?**
- Start with **filter methods** for quick screening.
- If you have a strong model and need high performance, try **wrapper methods**.
- **Embedded methods** are a good default because they balance speed and accuracy.

---

## 8. Conclusion

Today we learned:
- **Why feature selection is essential** – it reduces overfitting, improves accuracy, and speeds up training.
- **The three main categories** – Filter, Wrapper, Embedded (plus hybrids).
- **Practical code examples** for each category using Python and scikit-learn.

You can now apply these techniques to your own datasets. Remember, feature selection is often an iterative process – try different methods and validate using cross-validation.

**Homework**: Take a dataset (e.g., `load_diabetes` or any Kaggle dataset) and apply at least one filter, one wrapper, and one embedded method. Compare the number of selected features and the model performance. Which method worked best? Why do you think that is?

Happy coding! 🚀